# RAG Engine - C++ Build & Test

**IMPORTANT:**
1. Runtime > Change runtime type > **T4 GPU**
2. Runtime > Restart runtime
3. Run all cells in order

**Note:** This build uses CPU libraries. GPU acceleration will be tested via Python.

In [ ]:
# Setup
import os
import subprocess

PROJECT_DIR = '/content/rag-engine'
VCPKG_DIR = '/content/vcpkg'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir('/content')
print('Setup complete')

## 1. GPU Verification

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Install System Dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y cmake build-essential git curl wget unzip make ninja-build \
    libprotobuf-dev protobuf-compiler libuv1-dev g++ libomp-dev 2>&1 | tail -10
print('\nSystem packages installed')

## 3. Clone Repository

In [ ]:
!rm -rf rag-engine
!git clone https://github.com/gumeeee/rag-engine.git 2>&1 | tail -3
!mkdir -p /content/rag-engine/build /content/rag-engine/data/corpus /content/rag-engine/models
print('Repository cloned')

## 4. Install vcpkg

In [ ]:
if not os.path.exists('/content/vcpkg'):
    print('Cloning vcpkg...')
    !git clone https://github.com/Microsoft/vcpkg.git /content/vcpkg 2>&1 | tail -3
print('Bootstrapping...')
!cd /content/vcpkg && ./bootstrap-vcpkg.sh 2>&1 | tail -5
print('vcpkg ready')

## 5. Install C++ Dependencies

In [ ]:
!export VCPKG_ROOT=/content/vcpkg && cd /content/vcpkg
!/content/vcpkg/vcpkg install faiss-cpu onnxruntime libuv protobuf spdlog nlohmann-json --triplet x64-linux 2>&1 | tail -15
print('\nDependencies installed')

## 6. Generate Sample Corpus

In [ ]:
docs = {
    'transformer.txt': 'The Transformer uses self-attention mechanisms for sequence processing.',
    'bert.txt': 'BERT uses bidirectional transformers for language understanding.',
    'gpt.txt': 'GPT is an autoregressive language model using next-token prediction.',
    'rag.txt': 'RAG combines LLM with vector retrieval for accurate answers.',
    'vector_search.txt': 'Vector search uses embeddings. FAISS enables efficient search.'
}
for name, content in docs.items():
    with open(f'/content/rag-engine/data/corpus/{name}', 'w') as f:
        f.write(content)
print(f'Created {len(docs)} documents')

## 7. Generate Embeddings

In [ ]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer
import numpy as np

print('Loading model...')
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
texts = list(docs.values())
print('Generating embeddings...')
embeddings = model.encode(texts, convert_to_numpy=True)

output = '/content/rag-engine/data/corpus.corpus'
with open(output, 'wb') as f:
    f.write(len(texts).to_bytes(4, 'little'))
    f.write(embeddings.shape[1].to_bytes(4, 'little'))
    embeddings.astype(np.float32).tofile(f)
print(f'Saved {len(texts)} embeddings')

## 8. Build C++ Project

In [ ]:
import os
os.chdir('/content/rag-engine/build')

# Set environment
os.environ['VCPKG_ROOT'] = '/content/vcpkg'
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/make'

print('Configuring CMake...')
result = subprocess.run(
    'cmake .. -G "Unix Makefiles" -DCMAKE_BUILD_TYPE=Release '
    '-DCMAKE_TOOLCHAIN_FILE=/content/vcpkg/scripts/buildsystems/vcpkg.cmake '
    '-DVCPKG_TARGET_TRIPLET=x64-linux',
    shell=True, capture_output=True, text=True, timeout=300
)
print(result.stdout[-2000:] if result.stdout else 'No output')
if result.returncode != 0:
    print('\nERROR:', result.stderr[-1000:] if result.stderr else '')

In [ ]:
print('Building...')
result = subprocess.run(
    'make -j$(nproc)',
    shell=True, capture_output=True, text=True, timeout=1800
)
print(result.stdout[-3000:] if result.stdout else 'No output')
if result.returncode == 0:
    print('\n=== BUILD SUCCEEDED ===')
else:
    print('\n=== BUILD FAILED ===')
    print(result.stderr[-2000:] if result.stderr else '')

## 9. Verify Build

In [ ]:
import os
binary = '/content/rag-engine/build/rag-engine'
if os.path.exists(binary):
    size = os.path.getsize(binary) / 1024 / 1024
    print(f'Binary found: {binary}')
    print(f'Size: {size:.1f} MB')
    print('\nBuild successful!')
else:
    print('Build failed - see errors above')
    print('\nAlternative: Use Docker for full GPU testing')